### Make search better

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.dataset import load_clerc


clerc_data = load_clerc("train", limit=2000)

clerc_data

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

Dataset to pydantic transformation:   0%|          | 0/2000 [00:00<?, ?it/s]

ClercSplit(corpus=41167 docs, queries=2000, qrels=2000)

In [3]:
from qdrant_client import QdrantClient
import os 

inference_client = QdrantClient(
    url=os.getenv('QDRANT_INF_URL'),
    api_key=os.getenv('QDRANT_INF_API_KEY'),
    cloud_inference=True,
    timeout=300,
)

In [4]:
COLLECTION_NAME = 'clerc_cloud'

In [5]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance, Modifier

configs = [
    EmbeddingConfig(
        name='dense',   
        model_id='qwen/qwen3-embedding-8b',
        kind='dense',
        backend='openrouter',
        parallel=16,
        size=4096, 
        distance=Distance.COSINE,
        doc_options={
            "openrouter-api-key": os.getenv("OPEN_ROUTER_API_KEY"),
            'dimensions': 4096,
            "provider": {"sort": "latency"},
        },
        query_prompt=(
            "Instruct: Given a passage from a legal opinion with a citation removed, "
            "retrieve the cited case passage.\nQuery: "
        ),
    ),
    EmbeddingConfig(
        name='sparse',
        model_id='qdrant/bm25',
        kind='sparse',
        modifier=Modifier.IDF
    )
]

configs

[EmbeddingConfig(name='dense', model_id='qwen/qwen3-embedding-8b', kind='dense', backend='openrouter', size=4096, distance=<Distance.COSINE: 'Cosine'>, providers=None, parallel=16, device=None, query_prompt='Instruct: Given a passage from a legal opinion with a citation removed, retrieve the cited case passage.\nQuery: ', modifier=None, hnsw_m=None),
 EmbeddingConfig(name='sparse', model_id='qdrant/bm25', kind='sparse', backend='fastembed', size=None, distance=<Distance.COSINE: 'Cosine'>, providers=None, parallel=None, device=None, query_prompt=None, modifier=<Modifier.IDF: 'idf'>, hnsw_m=None)]

In [6]:
from src.indexing import EmbeddingCache
embedding_cache = EmbeddingCache()

In [7]:
from src.indexing import DocumentIndexer


document_vectors = DocumentIndexer(
    inference_client,
    COLLECTION_NAME,
    embeddings=configs,
    cache=embedding_cache
)

In [8]:
document_vectors.ensure_collection()

In [10]:
document_vectors.upload(
    items=list(clerc_data.corpus.values()), 
    batch_size=32, 
    skip_existing=True,
    parallel=10
)

upsert:   0%|          | 0/960 [00:00<?, ?it/s]

In [9]:
from src.search import HybridSearcher

searcher_hybrid = HybridSearcher(
    client=inference_client, 
    collection_name=COLLECTION_NAME, 
    embeddings=configs,
)

searcher_hybrid

In [9]:
searcher_hybrid.warm_up([q.query for q in clerc_data.queries.values()], parallel=16)

openrouter:qwen/qwen3-embedding-8b:   0%|          | 0/32 [00:00<?, ?it/s]

In [12]:
from src.evaluation import evaluate_retrieval

report_hybrid = evaluate_retrieval(searcher_hybrid, clerc_data)  # smoke run
print(report_hybrid)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

ResponseHandlingException: Server disconnected without sending a response.

In [24]:
searcher_hybrid.search(" there")

[SearchHit(doc_id='10852795', text="later and did not return to China until 1914, where he died April 16, 1915. It must be conceded that it would be unreasonable and unfair for the immigration authorities, after fully investigating the discrepancy between the statement of the alleged father in 1897, when he stated that he was unmarried and his later statement made in 1909 in an effort to secure the entry of his son Wong Woon, and having determined that Wong Woon was the legitimate son of the marriage of the father and Horn Shee, and after having reached a similar conclusion in 1924 on the admission of Wong Cheng, and alleged brother, to turn about and on the| same evidence and without any additional circumstances to hold that no such marriage occurred, and for that reason deny admission to the alleged second son, but we have here, the additional fact that the father stated that, his wife died in 1909, whereas, the applicant', and his witnesses ail testified that their mother is still l

In [10]:
from src.search import DenseSearcher

searcher_dense = DenseSearcher(
    client=inference_client, 
    collection_name=COLLECTION_NAME, 
    embedding=configs[0],
)

searcher_dense

In [15]:
searcher_dense.warm_up([q.query for q in clerc_data.queries.values()], parallel=16)

openrouter:qwen/qwen3-embedding-8b:   0%|          | 0/32 [00:00<?, ?it/s]

In [16]:
from src.evaluation import evaluate_retrieval

report_dense = evaluate_retrieval(searcher_dense, clerc_data)  # smoke run
print(report_dense)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-10):
  ndcg@10      0.1978
  recall@10    0.3975
  mrr@10       0.1383


In [20]:
from src.evaluation import evaluate_retrieval, RetrievalMetric

report_dense = evaluate_retrieval(
    searcher_dense, 
    clerc_data, 
    top_k=100, 
    limit=100,
    metrics=[
        RetrievalMetric.RECALL_100,
        RetrievalMetric.NDCG_100, 
        RetrievalMetric.MRR_100,
    ],
)  # smoke run
print(report_dense)

search:   0%|          | 0/100 [00:00<?, ?it/s]

Retrieval evaluation (100 queries, top-100):
  recall@100   0.8800
  ndcg@100     0.3061
  mrr@100      0.1638


### Rerank: Cohere Rerank 4 Pro (OpenRouter API) over a multiplied candidate pool

In [11]:
from src.rerank import OpenRouterRerankStrategy, Reranker

# retriever is a parameter — swap in any Searcher (searcher_dense, searcher_hybrid, ...)
searcher_reranked = Reranker(
    retriever=searcher_hybrid,
    strategy=OpenRouterRerankStrategy(api_key=os.getenv("OPEN_ROUTER_API_KEY")),
    multiplier=10,
)

searcher_reranked

In [12]:
# no-op if the hybrid retriever's query cache is already warm
searcher_reranked.warm_up([q.query for q in clerc_data.queries.values()], parallel=16)

openrouter:qwen/qwen3-embedding-8b:   0%|          | 0/32 [00:00<?, ?it/s]

In [14]:
from src.evaluation import evaluate_retrieval

# smoke run on 200 queries first (~200 rerank searches ~ $0.50);
# drop `limit` for the full 2000-query run (~$5)
report_reranked = evaluate_retrieval(searcher_reranked, clerc_data, limit=200)
print(report_reranked)

search:   0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Entity signal: can legal citations rerank what embeddings can't?

Hypothesis: the query's surviving legal identifiers (statute cites, reporter cites, section refs, case names — the *target* citation is removed by construction) discriminate the gold passage from topically-equivalent distractors. Check: rank gold among the query's own mined hard negatives by IDF-weighted entity overlap. If gold separates, an entity-only sparse rerank over the retrieved pool has real headroom; if not, the idea dies here for free.

In [15]:
from src.entity_analysis import analyze_entity_signal

entity_stats = analyze_entity_signal(clerc_data)
print(entity_stats)

entity-rank:   0%|          | 0/2000 [00:00<?, ?it/s]

Entity signal (2,000 queries, gold ranked among own hard negatives):
  Queries with >=1 entity:      1,842 (92.1%), median 6 entities/query
  Median pool size (gold+negs): 21
  Gold w/ zero entity overlap:  49.1%
  Mean relative rank of gold:   0.601 (random = 0.500)
  Gold in top-1 by entities:    4.1%
  Gold in top-3 by entities:    11.8%
  Gold in top-5 by entities:    17.2%
  Random top-1 baseline:        4.8%


In [17]:
for i in range(0, 100):
    sample = clerc_data.sample()
    print(sample[0], sample[1])

query_id='156097' query='it is clear that plaintiff will be unable to recover under any viable theory. Garita Hotel Ltd. Partnership v. Ponce Fed. Bank, 958 F.2d 15, 17 (1st Cir.1992). However, a court will not accept plaintiff’s “unsupported conclusions or interpretations of law”. Washington Legal Found v. Massachusetts Bar Found., 993 F.2d 962, 971 (1st Cir.1993). DORA H. GONZALEZ DOES NOT COMPLY WITH STANDING REQUIREMENTS In their opposition Plaintiff’s contend that Dora H. Gonzalez, the mother of the deceased, has standing to bring a § 1983 suit because Defendants’ actions were aimed at her family relationship with her son. The Court disagrees. There is no absolute constitutional right to enjoy the companionship of one’s family members free from all encroachments by the state.  REDACTED Valdivieso Ortiz v. Burgos, 807 F.2d 6, 8 (1st Cir.1986). The death of a family member will not ordinarily give those still alive a cognizable due process claim under § 1983. Serrano-Moran v. Toledo